In [2]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_distances

def compute_all_silhouette_scores(embeddings_matrix, df):
    print("\n" + "="*50)
    print("=== SILHOUETTE SCORES (SPAZIO 1024-D) ===")
    print("="*50)
    
    valid_sent = df['sentiment'].notna()
    score_sent = silhouette_score(embeddings_matrix[valid_sent], df.loc[valid_sent, 'sentiment'].values, metric='cosine')
    print(f"Silhouette Score (Sentiment): {score_sent:.4f}")
    
    valid_air = df['airline'].notna()
    score_air = silhouette_score(embeddings_matrix[valid_air], df.loc[valid_air, 'airline'].values, metric='cosine')
    print(f"Silhouette Score (Airline)  : {score_air:.4f}")
    
    valid_topic = (df['topic_id'].notna()) & (df['topic_id'] != -1)
    if valid_topic.sum() > 0:
        score_topic = silhouette_score(embeddings_matrix[valid_topic], df.loc[valid_topic, 'topic_id'].values, metric='cosine')
        print(f"Silhouette Score (Topic)    : {score_topic:.4f}")


def analyze_deep_metrics(embeddings_matrix, df, column_name, top_n=4):
    print("\n" + "="*50)
    print(f"=== ANALISI PROFONDA: {column_name.upper()} ===")
    print("="*50)
    
    valid_idx = (df[column_name].notna()) & (df[column_name] != -1)
    emb_valid = embeddings_matrix[valid_idx]
    labels = df.loc[valid_idx, column_name].values
    
    top_classes = df.loc[valid_idx, column_name].value_counts().nlargest(top_n).index
    
    print(f"\n--- Distanze Medie Intra-Cluster per '{column_name}' (Più basso = Più compatto) ---")
    class_embs = {}
    for cls in top_classes:
        cls_emb = emb_valid[labels == cls]
        class_embs[cls] = cls_emb
        if len(cls_emb) > 1:
            intra_dists = cosine_distances(cls_emb)
            mean_intra = np.sum(intra_dists) / (len(cls_emb)**2 - len(cls_emb))
            print(f"Compattezza '{cls}': {mean_intra:.4f}")
            
    print(f"\n--- Distanze Medie Inter-Cluster per '{column_name}' (Più alto = Più separato) ---")
    classes_list = list(top_classes)
    for i in range(len(classes_list)):
        for j in range(i+1, len(classes_list)):
            cls1, cls2 = classes_list[i], classes_list[j]
            emb1, emb2 = class_embs[cls1], class_embs[cls2]
            if len(emb1) > 0 and len(emb2) > 0:
                inter_dists = cosine_distances(emb1, emb2)
                mean_inter = np.mean(inter_dists)
                print(f"Distanza tra '{cls1}' e '{cls2}': {mean_inter:.4f}")


BASE_EMBEDDING_PATH = "/kaggle/input/datasets/claudiacornacchia/embedding"
csv_path = os.path.join(BASE_EMBEDDING_PATH, "base_map.csv")
embeddings_path = os.path.join(BASE_EMBEDDING_PATH, "embeddings_1024d.pt") 

print("Caricamento embedding e metadati in corso...")
embeddings_numpy = torch.load(embeddings_path, map_location='cpu').numpy()
df_dati = pd.read_csv(csv_path)
print(f"Dati caricati con successo. Matrice embedding shape: {embeddings_numpy.shape}")


compute_all_silhouette_scores(embeddings_numpy, df_dati)

analyze_deep_metrics(embeddings_numpy, df_dati, 'sentiment', top_n=3)

analyze_deep_metrics(embeddings_numpy, df_dati, 'airline', top_n=6)

analyze_deep_metrics(embeddings_numpy, df_dati, 'topic_id', top_n=13)

Caricamento embedding e metadati in corso...
Dati caricati con successo. Matrice embedding shape: (10445, 1024)

=== SILHOUETTE SCORES (SPAZIO 1024-D) ===
Silhouette Score (Sentiment): 0.0723
Silhouette Score (Airline)  : 0.0755
Silhouette Score (Topic)    : 0.0098

=== ANALISI PROFONDA: SENTIMENT ===

--- Distanze Medie Intra-Cluster per 'sentiment' (Più basso = Più compatto) ---
Compattezza 'negative': 0.2314
Compattezza 'neutral': 0.2619
Compattezza 'positive': 0.2372

--- Distanze Medie Inter-Cluster per 'sentiment' (Più alto = Più separato) ---
Distanza tra 'negative' e 'neutral': 0.2662
Distanza tra 'negative' e 'positive': 0.2579
Distanza tra 'neutral' e 'positive': 0.2650

=== ANALISI PROFONDA: AIRLINE ===

--- Distanze Medie Intra-Cluster per 'airline' (Più basso = Più compatto) ---
Compattezza 'United': 0.2482
Compattezza 'US Airways': 0.1855
Compattezza 'American': 0.2072
Compattezza 'Southwest': 0.2046
Compattezza 'Delta': 0.2041
Compattezza 'Virgin America': 0.2017

--- Di